<a href="https://githubtocolab.com/tannialhernandez/topicosAvanzadosAnalitica/blob/main/E8-TextSummary/E7_TextSummary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Text Summary

Text summarization is important in the field of machine learning and natural language processing for several reasons:

1. **Information Retrieval:** Text summarization helps users quickly grasp the main points or key information from a large document, making it easier to decide whether to read the full document or not. This is particularly valuable in scenarios where individuals are inundated with vast amounts of textual data, such as news articles, research papers, or social media posts.

2. **Time Efficiency:** Summarization algorithms can process and generate summaries much faster than humans can read and summarize large texts. This saves time and allows users to focus their attention on the most relevant content.

3. **Content Extraction:** Text summarization can automatically extract essential information from a document, enabling applications like content recommendation, keyword extraction, and topic modeling.

4. **Content Generation:** Summarization models can be used to generate concise, coherent, and informative summaries for various purposes, such as creating abstracts for research papers, news article headlines, or social media post previews.

5. **Multilingual Support:** Text summarization can be applied to texts in multiple languages, making it a valuable tool for global communication and information retrieval.

6. **Personalization:** Summarization can be personalized to individual preferences. Machine learning models can learn from user feedback to generate summaries that align more closely with a user's interests and priorities.

7. **Scalability:** As the volume of digital content continues to grow, automated summarization becomes crucial for scaling information processing and retrieval. Machine learning-based summarization models can adapt and handle large volumes of text efficiently.

8. **Legal and Compliance:** In legal and regulatory contexts, automated summarization can help organizations review contracts, policies, and legal documents to ensure compliance and identify critical clauses or information.

9. **Search Engine Optimization (SEO):** Summarized content can be used to create concise and engaging snippets for search engine results, improving the discoverability of web content.

10. **Content Creation:** Summarization can be integrated into content creation tools, helping authors and content creators generate concise and informative content more efficiently.

Overall, text summarization is an essential component of machine learning and natural language processing, enabling efficient information retrieval, content extraction, and content generation across a wide range of applications and industries. It plays a critical role in handling the ever-increasing amount of textual data available in the digital age.

---
Exercise:

Now, as a data scientist expert in NLP, you are asked to create a model to be able to summarize text in Spanish. Your stakeholders will pass you an article and your model should summarize it.

In [1]:
!pip install nltk
!pip install langdetect googletrans==3.1.0a0 sentence-transformers torch nltk
!pip install requests beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 11.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [2]:
import requests
from bs4 import BeautifulSoup

def obtain_article(url):
  # Realizar una solicitud HTTP para obtener el contenido de la página
  response = requests.get(url)

  # Verificar si la solicitud fue exitosa
  if response.status_code == 200:
      # Analizar el contenido HTML de la página con BeautifulSoup
      soup = BeautifulSoup(response.text, "html.parser")

      # Encontrar el contenido del artículo (puedes inspeccionar el HTML de la página para encontrar la estructura adecuada)
      article_content = soup.find("div", {"class": "article-content"})

      if not article_content:
        article_content = article_content = soup.find("article")

      # Extraer el texto del artículo
      article_text = ""
      for paragraph in article_content.find_all("p"):
          article_text += paragraph.get_text() + "\n"

      # Imprimir el texto del artículo

      print("\nArticulo completo:")
      print(article_text)
      return article_text
  else:
      print("Error al obtener la página:", response.status_code)

In [3]:
import re
import nltk
import torch
import numpy as np
from langdetect import detect
from googletrans import Translator
from nltk.tokenize import sent_tokenize
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer

In [4]:
nltk.download('punkt')
nltk.download('punkt_tab')

# Initialize the translator for translations
translator = Translator()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [5]:
 # Function to clean the text
def clean_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    # Replace multiple spaces and trim whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [6]:
def calculate_summary_length(original_text, min_ratio=0.1, max_ratio=0.3):
    text_length = len(original_text.split())  # Count words
    min_length = max(int(text_length * min_ratio), 50)  # Ensure at least 50 words
    max_length = max(int(text_length * max_ratio),
                     min_length)  # Ensure max_length is at least min_length
    return min_length, max_length

In [7]:
def translate_text(text):
  language = detect(text) if text.strip() else 'unknown'
  if language == 'en':
    return translator.translate(text, src='en',dest='es').text
  elif language == 'es':  # If Spanish
      return translator.translate(text, src='es', dest='en').text
  else:
      return "Unsupported language. Please provide English or Spanish text."

In [8]:
# Function to summarize text using BART
def summarize_text_bart(text):
    # Clean the text
    cleaned_text = clean_text(text)
    # Detect language of the text
    language = detect(cleaned_text) if cleaned_text.strip() else 'unknown'
    if language == 'en':
      translated = cleaned_text
    elif language == 'es':  # If Spanish
        translated = translate_text(text)
    else:
        return "Unsupported language. Please provide English or Spanish text."

    # Calculate appropriate lengths for summary
    min_length, max_length = calculate_summary_length(cleaned_text)

    # Load BART summarization model and tokenizer
    model_name = "facebook/bart-large-cnn"
    summarizer = pipeline("summarization", model=model_name, device=0 if torch.cuda.is_available() else -1)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Create the summarization pipeline using the loaded model and tokenizer
    inputs = tokenizer(translated, max_length=1024, truncation=True, return_tensors="pt").to(summarizer.device)

    # Generate summary using the tokenized input
    summary_ids = summarizer.model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_length,
        min_length=min_length,
        do_sample=False
    )


    # Generate summary using the tokenized input
    summary_ids = summarizer.model.generate(inputs["input_ids"],
                                           attention_mask=inputs["attention_mask"],
                                           max_length=max_length,
                                           min_length=min_length,
                                           do_sample=False)

    # Decode the summary IDs back into text
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    language = detect(summary) if summary.strip() else 'unknown'
    # Translate to Spanish if the detected language is English
    if language == 'en':  # If English
        summary_translated = translate_text(summary)
    elif language == 'es':  # If Spanish
        summary_translated = summary  # Use cleaned text as it is already in Spanish
    else:
        return "Unsupported language. Please provide English or Spanish text."

    return summary_translated

In [9]:
# Main function to translate and summarize text
def summarize_text_bert(text):
    # Clean the text
    cleaned_text = clean_text(text)

    # Detect language of the text
    language = detect(cleaned_text) if cleaned_text.strip() else 'unknown'

    # Translate to Spanish if the detected language is English
    if language == 'en':  # If English
        translated = translate_text(cleaned_text)
    elif language == 'es':  # If Spanish
        translated = cleaned_text  # Use cleaned text as it is already in Spanish
    else:
        return "Unsupported language. Please provide English or Spanish text."

    # Tokenize the translated text into sentences
    sentences = sent_tokenize(translated)

    # Load the BERT model for sentence embeddings
    model = SentenceTransformer('bert-base-nli-mean-tokens')

    # Generate sentence embeddings
    embeddings = model.encode(sentences)

    # Calculate scores for each sentence
    scores = np.linalg.norm(embeddings, axis=1)

    # Get indices of the top N sentences
    num_sentences = len(sentences)

    if num_sentences < 5:
        N = num_sentences
    elif 5 <= num_sentences <= 15:
        N = max(1, num_sentences // 2)
    else:
        N = 15


    top_sentence_indices = np.argsort(scores)[-N:]

    # Get the top sentences maintaining their original order
    top_sentences = [(i, sentences[i]) for i in sorted(top_sentence_indices)]

    # Reconstruct the summary
    summary_sentences = [sentence for i, sentence in top_sentences]
    summary = ' '.join(summary_sentences)

    return summary

In [10]:
# Call the summarization function

url = "https://time.com/collection/time100-ai/6309026/geoffrey-hinton/"
article_text = obtain_article(url)

summary = summarize_text_bart(article_text)
summary_bert = summarize_text_bert(article_text)


Articulo completo:
Over the course of February, Geoffrey Hinton, one of the most influential AI researchers of the past 50 years, had a “slow eureka moment.”
Hinton, 76, has spent his career trying to build AI systems that model the human brain, mostly in academia before joining Google in 2013. He had always believed that the brain was better than the machines that he and others were building, and that by making them more like the brain, they would improve. But in February, he realized “the digital intelligence we’ve got now may be better than the brain already. It’s just not scaled up quite as big.” 
Developers around the world are currently racing to build the biggest AI systems that they can. Given the current rate at which AI companies are increasing the size of models, it could be less than five years until AI systems have 100 trillion connections—roughly as many as there are between neurons in the human brain.
Alarmed, Hinton left his post as VP and engineering fellow in May and

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
print("Resumen BART:")
print(summary)

print("Resumen BERT:")
print(summary_bert)

Resumen BART:
Geoffrey Hinton es uno de los investigadores de IA más influyentes de los últimos 50 años. Hinton, de 76 años, ha pasado su carrera tratando de construir sistemas de IA que modelen el cerebro humano. Se preocupa por lo que podría suceder una vez que los sistemas de IA se reducen al tamaño de los cerebros humanos. "Si quieres saber cómo se siente, pregúntale a un pollo", dice Hinton sobre la posibilidad de que la humanidad sea eliminada por la tecnología que ayudó a crear. Dejó su puesto como vicepresidente y miembro de la ingeniería en Google en mayo para hablar libremente sobre los peligros de la IA.
Resumen BERT:
Hinton, de 76 años, ha pasado su carrera tratando de construir sistemas de IA que modelen el cerebro humano, principalmente en la academia antes de unirse a Google en 2013. Los desarrolladores de todo el mundo actualmente están corriendo para construir los sistemas de IA más grandes que puedan. Como estudiante universitario de la Universidad de Cambridge, probó

In [12]:
url = "https://www.bbva.com/es/sostenibilidad/biotecnologia-medica-un-futuro-fascinante-y-mas-saludable/"
articulo_es = obtain_article(url)

summary_es = summarize_text_bart(articulo_es)
summary_bert_es = summarize_text_bert(articulo_es)


Articulo completo:
La biotecnología médica aplica la ciencia y la tecnología a la medicina, consiguiendo que millones de personas en todo el mundo se beneficien de los medicamentos y los avances que se realizan en esta especialidad. Terapias génicas y regenerativas, medicina personalizada, xenotrasplantes o la técnica de las tijeras genéticas son algunas aportaciones de un campo que promete avances de ciencia ficción.
El gran público sabía poco de una disciplina multidisciplinar llamada biotecnología. Esta se puede definir como "la aplicación de la ciencia y la tecnología a los organismos vivos, así como a sus partes, productos y modelos, con el fin de alterar materiales vivos o no vivos para la producción de conocimientos, bienes y servicios". Así lo hace el documento 'A Framework for Biotechnology Statistics' de la Organización para la Cooperación y el Desarrollo Económico (OCDE).
Medioambiente
La biotecnología gris o biotecnología ambiental es una rama enfocada en la aplicación de 

Device set to use cpu


In [13]:
print("Resumen BART:")
print(summary_es)

print("Resumen BERT:")
print(summary_bert_es)

Resumen BART:
La biotecnología médica aplica ciencia y tecnología a la medicina. Las terapias genéricas y regenerativas, la medicina personalizada, los xenotransplantes o la técnica de las tijeras genéticas son algunas contribuciones. Las tecnologías como el análisis de "big data", la computación en la nube, el Internet de las cosas (IA) o la inteligencia artificial harán que los nuevos medicamentos sean más eficientes y caros. El futuro de la salud de las personas es, en gran parte, en biotecnología médica, dice la experta española Avanza Estrella Cortés, directora de la biotecnología de posgrado aplicada a la salud.
Resumen BERT:
Así lo hace el documento 'A Framework for Biotechnology Statistics' de la Organización para la Cooperación y el Desarrollo Económico (OCDE). Medioambiente La biotecnología gris o biotecnología ambiental es una rama enfocada en la aplicación de procesos biológicos para resolver problemas medioambientales. No obstante, otras aportaciones recientes de la biotec

# BART

BART es eficiente en la generación de textos coherentes y concisos, lo que lo hace ideal para crear resúmenes que capturan los puntos clave de un artículo.

Utiliza un enfoque de generación: toma un contexto completo y produce un resumen.
Captura relaciones a largo plazo en el texto, lo que le permite crear resúmenes que son más fluidos y menos propensos a perder el hilo narrativo.


# BERT

Esta función genera embeddings para oraciones y luego selecciona las oraciones más relevantes para construir un resumen, lo que puede resultar en un enfoque más fragmentado y menos coherente en comparación con BART.

Utiliza un enfoque de extracción: selecciona las oraciones más alineadas con el contenido original, basándose en los embeddings generados.
Puede producir resúmenes compuestos de oraciones que, aunque relevantes, a veces pueden carecer de la cohesión de un texto generado de forma más narrativa.


# Conclusión

La elección entre usar bart o bert depende del tipo de resumen que se desee producir. Si se busca un resumen más fluido y narrativo, el modelo BART es preferible. Es ideal para crear resúmenes que no solo contengan las ideas principales sino que también mantengan la coherencia del texto original.

Si el objetivo es ofrecer un resumen que preserve oraciones relevantes directamente del texto sin necesidad de generar oraciones completamente nuevas, el enfoque basado en BERT puede ser suficiente. Sin embargo, se debe tener en cuenta que el texto resultante puede no ser tan cohesivo.